In [ ]:
```python
!pip install crewai langchain-google-genai tavily-python pydantic pyyaml --quiet

import os
import yaml
from typing import List, Dict, Optional
from pydantic import BaseModel, Field
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from tavily import TavilyClient
from langchain_google_genai import ChatGoogleGenerativeAI

# Configure API keys and model
class Config:
    def __init__(self, model_name: str = "gemini-pro"):
        self.api_key = os.getenv("GEMINI_API_KEY", "your_gemini_api_key")
        self.tavily_key = os.getenv("TAVILY_API_KEY", "your_tavily_api_key")
        if not self.api_key or self.api_key == "your_gemini_api_key":
            raise EnvironmentError("GEMINI_API_KEY not set")
        if not self.tavily_key or self.tavily_key == "your_tavily_api_key":
            raise EnvironmentError("TAVILY_API_KEY not set")
        self.model_name = model_name
        self.tavily_client = TavilyClient(api_key=self.tavily_key)

    def get_llm(self):
        return ChatGoogleGenerativeAI(
            google_api_key=self.api_key,
            model=self.model_name,
            temperature=0.5,  # Changed from 0.7
            max_tokens=1200   # Added token limit
        )

# Data models
class Route(BaseModel):
    origin: str = Field(description="Starting point")
    destination: str = Field(description="End point")
    distance_km: float = Field(description="Distance in kilometers")
    estimated_time: str = Field(description="Estimated travel time")
    cost: float = Field(description="Estimated transport cost")

class RoutePlan(BaseModel):
    logistics_type: str = Field(description="Type: truck, rail, air")
    routes: List[Route] = Field(description="Optimized routes")

class CostBreakdown(BaseModel):
    logistics_type: str = Field(description="Type: truck, rail, air")
    fuel_cost: float = Field(description="Fuel cost")
    labor_cost: float = Field(description="Labor cost")
    maintenance_cost: float = Field(description="Maintenance cost")
    total_cost: float = Field(description="Total cost")

class DeliverySchedule(BaseModel):
    logistics_type: str = Field(description="Type: truck, rail, air")
    schedule: List[str] = Field(description="Delivery time slots")
    priority: str = Field(description="Priority: high, medium, low")

# Tools
@tool("route_optimization")
def route_optimization(logistics_type: str, origin: str, destination: str) -> str:
    """Simulate route optimization for logistics."""
    sample_routes = {
        "truck": {
            "mumbai-delhi": {"distance_km": 1400, "time": "24 hours", "cost": 15000},
            "delhi-chennai": {"distance_km": 2200, "time": "36 hours", "cost": 22000},
            "bangalore-kolkata": {"distance_km": 1800, "time": "30 hours", "cost": 18000}
        },
        "rail": {
            "mumbai-delhi": {"distance_km": 1380, "time": "20 hours", "cost": 10000},
            "delhi-chennai": {"distance_km": 2180, "time": "28 hours", "cost": 14000},
            "bangalore-kolkata": {"distance_km": 1870, "time": "26 hours", "cost": 12000}
        },
        "air": {
            "mumbai-delhi": {"distance_km": 1150, "time": "2 hours", "cost": 30000},
            "delhi-chennai": {"distance_km": 1760, "time": "2.5 hours", "cost": 35000},
            "bangalore-kolkata": {"distance_km": 1560, "time": "2.3 hours", "cost": 32000}
        }
    }
    key = f"{origin.lower()}-{destination.lower()}"
    routes = sample_routes.get(logistics_type.lower(), {})
    route = routes.get(key, {"distance_km": 1000, "time": "Unknown", "cost": 10000})
    return f"Route: {origin} to {destination}, Distance: {route['distance_km']} km, Time: {route['time']}, Cost: ₹{route['cost']}"

@tool("cost_research")
def cost_research(query: str) -> str:
    """Search for logistics cost data using Tavily."""
    try:
        config = Config()
        results = config.tavily_client.search(
            query=query,
            search_depth="basic",  # Changed from advanced
            max_results=2          # Changed from 5
        )
        formatted = []
        for r in results.get('results', []):
            formatted.extend([
                f"Source: {r.get('title', 'Unknown')}",
                f"Link: {r.get('url', 'Unknown')}",
                f"Info: {r.get('content', 'No content')[:100]}...",
                "---"
            ])
        return "\n".join(formatted)
    except Exception as e:
        return f"Cost research failed: {str(e)}"

# Agents
route_optimizer = Agent(
    role="Route Planner",
    goal="Optimize logistics routes for efficiency",
    backstory="Expert in finding the shortest, cost-effective routes for various transport modes.",
    tools=[route_optimization],
    verbose=True,
    llm=Config().get_llm()
)

cost_analyzer = Agent(
    role="Cost Estimator",
    goal="Estimate and analyze logistics costs",
    backstory="Skilled in breaking down fuel, labor, and maintenance costs for logistics operations.",
    tools=[cost_research],
    verbose=True,
    llm=Config().get_llm()
)

schedule_manager = Agent(
    role="Delivery Coordinator",
    goal="Create efficient delivery schedules",
    backstory="Experienced in scheduling deliveries to meet time-sensitive demands.",
    verbose=True,
    llm=Config().get_llm()
)

# Tasks
def create_route_task(logistics_type: str, origin: str, destination: str) -> Task:
    return Task(
        description=f"""
        Optimize a route for {logistics_type} from {origin} to {destination}.
        Provide:
        1. Distance and estimated time
        2. Transport cost
        3. Route feasibility for {logistics_type}
        """,
        expected_output="Optimized route details with distance, time, and cost",
        agent=route_optimizer,
        output_pydantic=RoutePlan
    )

def create_cost_task(logistics_type: str, origin: str, destination: str) -> Task:
    return Task(
        description=f"""
        Analyze costs for {logistics_type} from {origin} to {destination}.
        Include:
        1. Fuel cost
        2. Labor cost
        3. Maintenance cost
        4. Total cost
        Use external data if needed.
        """,
        expected_output="Detailed cost breakdown for logistics",
        agent=cost_analyzer,
        output_pydantic=CostBreakdown
    )

def create_schedule_task(logistics_type: str, priority: str) -> Task:
    return Task(
        description=f"""
        Create a delivery schedule for {logistics_type} with {priority} priority.
        Provide:
        1. Time slots for deliveries
        2. Priority level considerations
        Ensure schedules align with operational constraints.
        """,
        expected_output="Delivery schedule with time slots and priority",
        agent=schedule_manager,
        output_pydantic=DeliverySchedule
    )

# Main execution
def run_logistics_optimizer():
    print("=== Logistics Optimization System ===")
    logistics_type = input("Enter logistics type (truck/rail/air): ").strip().lower()
    origin = input("Enter origin city: ").strip()
    destination = input("Enter destination city: ").strip()
    priority = input("Enter priority (high/medium/low): ").strip().lower()

    if logistics_type not in ["truck", "rail", "air"]:
        logistics_type = "truck"
        print("Invalid logistics type. Defaulting to 'truck'.")
    if priority not in ["high", "medium", "low"]:
        priority = "medium"
        print("Invalid priority. Defaulting to 'medium'.")

    print(f"\n🔍 Optimizing logistics for: {logistics_type} from {origin} to {destination} ({priority} priority)")
    print("Processing...")

    route_task = create_route_task(logistics_type, origin, destination)
    cost_task = create_cost_task(logistics_type, origin, destination)
    schedule_task = create_schedule_task(logistics_type, priority)

    crew = Crew(
        agents=[route_optimizer, cost_analyzer, schedule_manager],
        tasks=[route_task, cost_task, schedule_task],
        process=Process.sequential,
        verbose=True
    )

    try:
        results = crew.kickoff()
        output = {
            "logistics_type": logistics_type,
            "origin": origin,
            "destination": destination,
            "priority": priority,
            "route_plan": route_task.output.raw,
            "cost_breakdown": cost_task.output.raw,
            "delivery_schedule": schedule_task.output.raw
        }

        # Save output as YAML
        with open("logistics_plan.yaml", "w") as f:
            yaml.dump(output, f, allow_unicode=True)

        print("\n" + "="*50)
        print("✅ Logistics Plan Generated!")
        print("="*50)
        print(f"🚚 Route: {origin} to {destination} via {logistics_type}")
        print(f"📅 Priority: {priority}")
        print("📄 Plan saved to logistics_plan.yaml")

        return output

    except Exception as e:
        print(f"System error: {e}")
        return None

# Simplified fallback
def basic_logistics_optimizer():
    print("=== Basic Logistics Optimizer ===")
    logistics_type = "truck"
    origin = "Mumbai"
    destination = "Delhi"
    priority = "medium"

    route_data = route_optimization(logistics_type, origin, destination)
    output = {
        "logistics_type": logistics_type,
        "origin": origin,
        "destination": destination,
        "priority": priority,
        "route_plan": route_data,
        "cost_breakdown": "Fuel: ₹8000, Labor: ₹5000, Maintenance: ₹2000, Total: ₹15000",
        "delivery_schedule": "Day 1: Morning departure, Day 2: Afternoon delivery"
    }

    with open("basic_logistics_plan.yaml", "w") as f:
        yaml.dump(output, f, allow_unicode=True)

    print(f"\n🚚 Route: {origin} to {destination} via {logistics_type}")
    print(f"📅 Priority: {priority}")
    print(f"📄 Saved to basic_logistics_plan.yaml")
    return output

if __name__ == "__main__":
    try:
        run_logistics_optimizer()
    except Exception as e:
        print(f"CrewAI error: {e}")
        print("Switching to basic optimizer...")
        basic_logistics_optimizer()
```